# Effect of TPE Optimisation on GRU Accuracy Relative to Random Walk Forecasts of IHSG
Notebook evaluasi untuk artikel bit-Tech. Notebook ini **tidak mengulang pipeline skripsi (script 01-19)**. Notebook ini memakai file hasil pipeline tersebut, lalu menjalankan semua analisis tambahan untuk jurnal:

1. Analisis pembanding random walk, uji Diebold-Mariano, dekomposisi error, akurasi arah, segmen tahun dan volatilitas, serta Gambar 1-3 (script 20)
2. Pencarian TPE untuk LSTM dengan ruang pencarian dan budget yang sama dengan GRU (script 21)
3. Pelatihan ulang 10 seed untuk semua konfigurasi (script 22)
4. Pembanding ARIMA (script 23)
5. Evaluasi walk-forward (script 24)
6. Rekap ketahanan hasil dan uji DM dengan koreksi Holm (script 25)

**File yang harus ada di folder yang sama:** `final_data.csv`, `train_data.csv`, `test_data.csv`, `X_train.npy`, `y_train.npy`, `baseline_predictions.csv`, `optimized_gru_predictions.csv`, `optuna_gru_results.csv`.

**Perkiraan waktu (CPU 2 core):** bagian 1, 4, dan 6 hanya beberapa detik sampai menit; bagian 2 sekitar 30-60 menit; bagian 3 sekitar 30 menit; bagian 5 sekitar 20 menit. Hasil tiap bagian disimpan ke CSV, jadi bagian yang berat cukup dijalankan sekali.

## 0. Instalasi dan fungsi bersama

In [1]:
# Jalankan sekali jika library belum terpasang
# !pip install tensorflow==2.21.0 keras==3.15.1 optuna==5.0.0 statsmodels==0.15.0 scikit-learn scipy matplotlib pandas numpy

In [2]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GRU, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler, StandardScaler

WINDOW = 30
SEEDS = [42, 7, 13, 21, 99, 123, 256, 512, 777, 2024]

# Manually set configuration (scripts 10 and 11) and GRU-TPE configuration (trial 51 of 13_optuna_gru.py)
CONFIGS = {
    "LSTM":    dict(cell="LSTM", units=64, dropout=0.2, lr=0.001, batch=64),
    "GRU":     dict(cell="GRU",  units=64, dropout=0.2, lr=0.001, batch=64),
    "GRU-TPE": dict(cell="GRU",  units=64, dropout=0.1, lr=0.0018829770816374, batch=32),
}


def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def load_series():
    df = pd.read_csv("final_data.csv")
    df["Date"] = pd.to_datetime(df["Date"])
    return df.sort_values("Date").reset_index(drop=True)


def make_level_windows(df, train_end, test_start, test_end, use_djia=True):
    """Level formulation used in the thesis: Min-Max scaled IHSG_Close and DJIA_Lag1,
    scalers fitted on rows [0, train_end). Test samples cover rows [test_start, test_end)."""
    cols = ["IHSG_Close", "DJIA_Lag1"] if use_djia else ["IHSG_Close"]
    scalers = {c: MinMaxScaler().fit(df.loc[:train_end - 1, [c]]) for c in cols}
    scaled = np.column_stack([scalers[c].transform(df[[c]]).ravel() for c in cols])
    target = scaled[:, 0]
    X_tr = np.array([scaled[i - WINDOW:i] for i in range(WINDOW, train_end)])
    y_tr = target[WINDOW:train_end]
    X_te = np.array([scaled[i - WINDOW:i] for i in range(test_start, test_end)])
    inverse = lambda v: scalers["IHSG_Close"].inverse_transform(np.asarray(v).reshape(-1, 1)).ravel()
    return X_tr, y_tr, X_te, inverse


def make_return_windows(df, train_end, test_start, test_end):
    """Return formulation: inputs are daily returns of IHSG and DJIA_Lag1 (standardised on the
    training rows), the target is the next-day IHSG return; levels are rebuilt as y_{t-1}(1 + r_hat)."""
    r = df[["IHSG_Close", "DJIA_Lag1"]].pct_change()
    r.iloc[0] = 0.0
    scaler = StandardScaler().fit(r.iloc[1:train_end])
    scaled = scaler.transform(r)
    X_tr = np.array([scaled[i - WINDOW:i] for i in range(WINDOW + 1, train_end)])
    y_tr = scaled[WINDOW + 1:train_end, 0]
    X_te = np.array([scaled[i - WINDOW:i] for i in range(test_start, test_end)])
    mu, sd = scaler.mean_[0], scaler.scale_[0]
    prev = df["IHSG_Close"].values[test_start - 1:test_end - 1]
    to_level = lambda v: prev * (1 + (np.asarray(v).ravel() * sd + mu))
    return X_tr, y_tr, X_te, to_level


def build_model(cfg, n_features):
    cell = GRU if cfg["cell"] == "GRU" else LSTM
    model = Sequential([Input(shape=(WINDOW, n_features)), cell(cfg["units"]), Dropout(cfg["dropout"]), Dense(1)])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=cfg["lr"]), loss="mse")
    return model


def train_predict(cfg, X_tr, y_tr, X_te, seed):
    set_seed(seed)
    model = build_model(cfg, X_tr.shape[2])
    history = model.fit(X_tr, y_tr, validation_split=0.2, epochs=100, batch_size=cfg["batch"],
                        callbacks=[EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)],
                        verbose=0)
    return model.predict(X_te, verbose=0).ravel(), min(history.history["val_loss"])

## 1. Analisis pembanding random walk (script 20)
Menghasilkan `article_accuracy.csv`, `article_dm_tests.csv`, `article_change_diagnostics.csv`, `article_mse_decomposition.csv`, `article_segments.csv`, dan tiga gambar artikel.

In [3]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ==========================================
# 1. LOAD DATA
# ==========================================

baseline = pd.read_csv("baseline_predictions.csv")
optimized = pd.read_csv("optimized_gru_predictions.csv")
train_data = pd.read_csv("train_data.csv")
test_data = pd.read_csv("test_data.csv")
optuna_results = pd.read_csv("optuna_gru_results.csv")

y = baseline["Actual_IHSG"].values
assert np.allclose(y, test_data["IHSG_Close"].values), "Actual values do not match test_data.csv"

# Random-walk benchmark: forecast for day t is the actual value of day t-1.
# The first test day uses the last training observation.
previous_actual = np.r_[train_data["IHSG_Close"].values[-1], y[:-1]]

forecasts = {
    "Random walk": previous_actual,
    "LSTM": baseline["LSTM_Prediction"].values,
    "GRU": baseline["GRU_Prediction"].values,
    "GRU-TPE": optimized["Predicted_IHSG"].values,
}
models = ["LSTM", "GRU", "GRU-TPE"]
n = len(y)


# ==========================================
# 2. ACCURACY METRICS
# ==========================================

def rmse(e):
    return np.sqrt(np.mean(e ** 2))

rw_rmse = rmse(y - previous_actual)

rows = []
for name, pred in forecasts.items():
    e = y - pred
    rows.append({
        "Model": name,
        "RMSE": rmse(e),
        "MAE": np.mean(np.abs(e)),
        "MAPE (%)": 100 * np.mean(np.abs(e / y)),
        "R2": 1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2),
        "RMSE / RMSE_RW": rmse(e) / rw_rmse,
        "Mean error": e.mean(),
    })
accuracy = pd.DataFrame(rows)
print("\nACCURACY ON THE TEST SET (n = %d)" % n)
print(accuracy.round(4).to_string(index=False))
accuracy.to_csv("article_accuracy.csv", index=False)


# ==========================================
# 3. DIEBOLD-MARIANO TESTS
# ==========================================

def diebold_mariano(actual, pred_a, pred_b):
    """Positive statistic: model B is more accurate than model A."""
    d = (actual - pred_a) ** 2 - (actual - pred_b) ** 2
    m = len(d)
    statistic = d.mean() / np.sqrt(np.var(d) / m)
    p_value = 2 * (1 - stats.t.cdf(abs(statistic), df=m - 1))
    return statistic, p_value

pairs = [
    ("LSTM", "GRU"), ("GRU", "GRU-TPE"), ("LSTM", "GRU-TPE"),
    ("Random walk", "LSTM"), ("Random walk", "GRU"), ("Random walk", "GRU-TPE"),
]
dm_rows = []
for a, b in pairs:
    s, p = diebold_mariano(y, forecasts[a], forecasts[b])
    dm_rows.append({"Model A": a, "Model B": b, "DM": s, "p-value": p})
dm_table = pd.DataFrame(dm_rows)
print("\nDIEBOLD-MARIANO TESTS")
print(dm_table.round(4).to_string(index=False))
dm_table.to_csv("article_dm_tests.csv", index=False)


# ==========================================
# 4. CHANGE-BASED DIAGNOSTICS AND DIRECTIONAL ACCURACY
# ==========================================

def wilson_interval(k, m, z=1.959964):
    p = k / m
    centre = (p + z ** 2 / (2 * m)) / (1 + z ** 2 / m)
    half = z * np.sqrt(p * (1 - p) / m + z ** 2 / (4 * m ** 2)) / (1 + z ** 2 / m)
    return 100 * (centre - half), 100 * (centre + half)

actual_change = y - previous_actual
diag_rows = []
for name in models:
    pred_change = forecasts[name] - previous_actual
    # directional accuracy from the second test day onward (n - 1 days)
    correct = (np.sign(pred_change[1:]) == np.sign(actual_change[1:]))
    k, m = int(correct.sum()), len(correct)
    lo, hi = wilson_interval(k, m)
    diag_rows.append({
        "Model": name,
        "Mean |predicted change|": np.mean(np.abs(pred_change)),
        "Corr(predicted change, actual change)": np.corrcoef(pred_change, actual_change)[0, 1],
        "Correct directions": k,
        "Days": m,
        "DA (%)": 100 * k / m,
        "CI low (%)": lo,
        "CI high (%)": hi,
        "Binomial p (vs 50%)": stats.binomtest(k, m, 0.5).pvalue,
    })
diagnostics = pd.DataFrame(diag_rows)

# MSE decomposition relative to the random walk:
# MSE_model = MSE_RW + E[pc^2] - 2 E[dy * pc], where pc = predicted change, dy = actual change
dec_rows = []
for name in models:
    pc = forecasts[name] - previous_actual
    dec_rows.append({
        "Model": name,
        "MSE_RW": np.mean(actual_change ** 2),
        "E[pc^2]": np.mean(pc ** 2),
        "2E[dy*pc]": 2 * np.mean(actual_change * pc),
        "MSE_model": np.mean((y - forecasts[name]) ** 2),
    })
decomposition = pd.DataFrame(dec_rows)
print("\nMSE DECOMPOSITION RELATIVE TO THE RANDOM WALK")
print(decomposition.round(1).to_string(index=False))
decomposition.to_csv("article_mse_decomposition.csv", index=False)
print("\nMean |actual change|: %.2f index points" % np.mean(np.abs(actual_change)))
print("CHANGE-BASED DIAGNOSTICS")
print(diagnostics.round(4).to_string(index=False))
diagnostics.to_csv("article_change_diagnostics.csv", index=False)


# ==========================================
# 5. RMSE BY YEAR AND BY VOLATILITY REGIME
# ==========================================

dates = pd.to_datetime(test_data["Date"]).values
years = pd.DatetimeIndex(dates).year.values

returns = pd.Series(y).pct_change()
volatility = returns.rolling(20).std()          # sample std (ddof = 1)
median_vol = volatility.median()
has_vol = volatility.notna().values
high = has_vol & (volatility.values > median_vol)
low = has_vol & (volatility.values <= median_vol)

segments = [("Year 2023", years == 2023), ("Year 2024", years == 2024), ("Year 2025", years == 2025),
            ("Low volatility", low), ("High volatility", high)]
seg_rows = []
for label, mask in segments:
    row = {"Segment": label, "n": int(mask.sum())}
    for name, pred in forecasts.items():
        row[name] = rmse((y - pred)[mask])
    seg_rows.append(row)
segment_table = pd.DataFrame(seg_rows)
print("\nMedian 20-day volatility: %.4f%%" % (100 * median_vol))
print("RMSE BY SEGMENT")
print(segment_table.round(2).to_string(index=False))
segment_table.to_csv("article_segments.csv", index=False)


# ==========================================
# 6. TPE SEARCH SUMMARY
# ==========================================

opt = optuna_results.copy()
opt["duration_s"] = pd.to_timedelta(opt["duration"]).dt.total_seconds()
total_time = pd.to_datetime(opt["datetime_complete"]).max() - pd.to_datetime(opt["datetime_start"]).min()
best = opt.loc[opt["value"].idxmin()]
print("\nTPE SEARCH")
print("Total time:", total_time, "| mean trial: %.1f s" % opt["duration_s"].mean())
print("Best trial:", int(best["number"]), "value: %.9f" % best["value"])
print("Top-10 trials within %.2f%% of the best value"
      % (100 * (opt["value"].nsmallest(10).max() / opt["value"].min() - 1)))
for col in ["params_units", "params_dropout_rate", "params_batch_size"]:
    print(opt.groupby(col)["value"].agg(["count", "median", "min"]).to_string())
opt["lr_range"] = pd.cut(opt["params_learning_rate"], [1e-4, 1e-3, 3e-3, 1e-2])
print(opt.groupby("lr_range", observed=False)["value"].agg(["count", "median", "min"]).to_string())


# ==========================================
# 7. FIGURES
# ==========================================

plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 9, "axes.linewidth": 0.8})
GREY, SLATE, RED, BLACK = "#9ca0a8", "#4a5468", "#c0392b", "#222222"
colours = {"Random walk": BLACK, "LSTM": GREY, "GRU": SLATE, "GRU-TPE": RED}

# ---- Figure 1: objective value by hyperparameter value
fig, axes = plt.subplots(1, 4, figsize=(6.3, 2.2), dpi=300, sharey=True)
jitter = np.random.default_rng(0)
vals = opt["value"] * 1e4
for ax, col, label in zip(axes[:3],
                          ["params_units", "params_dropout_rate", "params_batch_size"],
                          ["Units", "Dropout rate", "Batch size"]):
    levels = sorted(opt[col].unique())
    for i, lv in enumerate(levels):
        sel = opt[col] == lv
        ax.scatter(i + jitter.uniform(-0.18, 0.18, sel.sum()), vals[sel], s=7, color=GREY, alpha=0.8, lw=0)
        ax.hlines(np.median(vals[sel]), i - 0.3, i + 0.3, color=BLACK, lw=1.2)
    ax.set_xticks(range(len(levels)), [("%g" % lv) for lv in levels])
    ax.set_xlabel(label)
ax = axes[3]
ax.scatter(opt["params_learning_rate"], vals, s=7, color=GREY, alpha=0.8, lw=0)
ax.scatter(best["params_learning_rate"], best["value"] * 1e4, s=22, facecolor="none", edgecolor=RED, lw=1.2)
ax.set_xscale("log"); ax.set_xlabel("Learning rate")
axes[0].set_ylabel("Validation loss (×10⁻⁴)")
axes[0].set_ylim(2.0, 4.6)
for a in axes:
    a.grid(axis="y", ls=":", lw=0.6, color="#bbbbbb")
    a.tick_params(length=3, width=0.8, labelsize=8)
fig.tight_layout(w_pad=0.6)
fig.savefig("fig_tpe_landscape.png", dpi=300)
plt.close(fig)

# ---- Figure 2: predicted change versus actual change
fig, axes = plt.subplots(1, 3, figsize=(6.3, 2.35), dpi=300, sharex=True, sharey=True)
lim = 260
for ax, name in zip(axes, models):
    pc = forecasts[name] - previous_actual
    ax.scatter(actual_change, pc, s=5, color=colours[name], alpha=0.6, lw=0)
    ax.axhline(0, color="#888888", lw=0.6); ax.axvline(0, color="#888888", lw=0.6)
    ax.plot([-lim, lim], [-lim, lim], ls="--", lw=0.7, color=BLACK)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("Actual change (points)")
    ax.tick_params(length=3, width=0.8, labelsize=8)
axes[0].set_ylabel("Predicted change (points)")
fig.tight_layout(w_pad=0.6)
fig.savefig("fig_change_scatter.png", dpi=300)
plt.close(fig)

# ---- Figure 3: RMSE by year and by volatility regime, including the random walk
order = ["Random walk", "LSTM", "GRU", "GRU-TPE"]
fig, (a, b) = plt.subplots(1, 2, figsize=(6.3, 2.5), dpi=300,
                           gridspec_kw={"width_ratios": [3, 2]})
w = 0.2
x = np.arange(3)
for i, k in enumerate(order):
    a.bar(x + (i - 1.5) * w, segment_table[k].values[:3], w, color=colours[k], label=k, zorder=3)
a.set_xticks(x, ["2023", "2024", "2025"]); a.set_xlabel("Test year")
a.set_ylabel("RMSE (index points)"); a.set_title("(a) By test year", fontsize=9)
a.legend(frameon=False, fontsize=7.5, loc="upper left", ncol=2)
a.set_ylim(0, 125)
x2 = np.arange(2)
for i, k in enumerate(order):
    b.bar(x2 + (i - 1.5) * w, segment_table[k].values[3:], w, color=colours[k], zorder=3)
b.set_xticks(x2, ["Low volatility", "High volatility"]); b.set_title("(b) By volatility regime", fontsize=9)
b.set_ylim(0, 125)
for ax in (a, b):
    ax.grid(axis="y", ls=":", lw=0.6, color="#bbbbbb", zorder=0)
    ax.tick_params(length=3, width=0.8, labelsize=8)
fig.tight_layout(w_pad=1.5)
fig.savefig("fig_segments.png", dpi=300)
plt.close(fig)

print("\nSaved: article_accuracy.csv, article_dm_tests.csv, article_change_diagnostics.csv,")
print("       article_segments.csv, fig_tpe_landscape.png, fig_change_scatter.png, fig_segments.png")


ACCURACY ON THE TEST SET (n = 518)
      Model    RMSE     MAE  MAPE (%)     R2  RMSE / RMSE_RW  Mean error
Random walk 72.7421 53.5723    0.7438 0.9814          1.0000      3.2935
       LSTM 84.7261 63.7552    0.8858 0.9748          1.1647      3.1187
        GRU 81.4721 60.9238    0.8468 0.9767          1.1200      2.9160
    GRU-TPE 73.7028 54.4745    0.7555 0.9809          1.0132      7.9580

DIEBOLD-MARIANO TESTS
    Model A Model B      DM  p-value
       LSTM     GRU  2.5705   0.0104
        GRU GRU-TPE  3.5214   0.0005
       LSTM GRU-TPE  3.9422   0.0001
Random walk    LSTM -4.1353   0.0000
Random walk     GRU -3.7006   0.0002
Random walk GRU-TPE -1.7706   0.0772

MSE DECOMPOSITION RELATIVE TO THE RANDOM WALK
  Model  MSE_RW  E[pc^2]  2E[dy*pc]  MSE_model
   LSTM  5291.4   2460.5      573.4     7178.5
    GRU  5291.4   1688.3      342.0     6637.7
GRU-TPE  5291.4    119.3      -21.4     5432.1

Mean |actual change|: 53.57 index points
CHANGE-BASED DIAGNOSTICS
  Model  Mean |

## 2. Pencarian TPE untuk LSTM (script 21)
Ruang pencarian, fungsi objektif, 100 trial, dan seed sampler sama dengan `13_optuna_gru.py`. Hasil: `optuna_lstm_results.csv`. Sel ini otomatis dilewati jika file hasilnya sudah ada.

In [4]:
import os
RUN_LSTM_SEARCH = not os.path.exists("optuna_lstm_results.csv")
print("Jalankan pencarian LSTM:", RUN_LSTM_SEARCH)

Jalankan pencarian LSTM: False


In [5]:
import numpy as np
import optuna
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
set_seed(SEED)
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")


def objective(trial):
    cfg = dict(cell="LSTM",
               units=trial.suggest_int("units", 32, 128, step=32),
               dropout=trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1),
               lr=trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
               batch=trial.suggest_categorical("batch_size", [16, 32, 64]))
    # as in 13_optuna_gru.py, the seed is set once at the start, not inside each trial
    model = build_model(cfg, X_train.shape[2])
    history = model.fit(X_train, y_train, validation_split=0.2, epochs=100, batch_size=cfg["batch"],
                        callbacks=[EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)],
                        verbose=0)
    return min(history.history["val_loss"])


if RUN_LSTM_SEARCH:
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=100)
    print("Best trial:", study.best_trial.number, "value:", study.best_value, study.best_params)
    study.trials_dataframe().to_csv("optuna_lstm_results.csv", index=False)

d:\Skripsi\gru-tpe-ihsg_repo\gru-tpe-ihsg\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Pelatihan ulang 10 seed (script 22)
Daftar konfigurasi diatur di variabel `MODELS`. Hasil ditambahkan ke `multiseed_predictions.csv`, jadi hapus file itu dulu jika ingin mengulang dari awal.

In [6]:
import os
import sys
import time
import pandas as pd

TRAIN_END, TEST_END = 2069, 2587
df = load_series()
dates = df["Date"].iloc[TRAIN_END:TEST_END].dt.strftime("%Y-%m-%d").values
actual = df["IHSG_Close"].iloc[TRAIN_END:TEST_END].values


def config_for(name):
    if name == "LSTM-TPE":
        res = pd.read_csv("optuna_lstm_results.csv")
        best = res.loc[res["value"].idxmin()]
        return dict(cell="LSTM", units=int(best["params_units"]), dropout=float(best["params_dropout_rate"]),
                    lr=float(best["params_learning_rate"]), batch=int(best["params_batch_size"]))
    return CONFIGS[name.replace("-noDJIA", "").replace("-return", "")]


out_file = "multiseed_predictions.csv"
MODELS = ["LSTM", "LSTM-TPE", "GRU", "GRU-TPE", "GRU-TPE-noDJIA", "GRU-TPE-return"]
for name in MODELS:
    cfg = config_for(name)
    if name.endswith("-return"):
        X_tr, y_tr, X_te, back = make_return_windows(df, TRAIN_END, TRAIN_END, TEST_END)
    else:
        X_tr, y_tr, X_te, back = make_level_windows(df, TRAIN_END, TRAIN_END, TEST_END,
                                                    use_djia=not name.endswith("-noDJIA"))
    for seed in SEEDS:
        t0 = time.time()
        pred, val = train_predict(cfg, X_tr, y_tr, X_te, seed)
        rows = pd.DataFrame({"model": name, "seed": seed, "Date": dates, "actual": actual,
                             "pred": back(pred), "val_loss": val})
        rows.to_csv(out_file, mode="a", header=not os.path.exists(out_file), index=False)
        print(name, seed, "%.1fs" % (time.time() - t0), flush=True)

LSTM 42 26.0s
LSTM 7 6.8s
LSTM 13 25.3s
LSTM 21 25.0s
LSTM 99 7.7s
LSTM 123 11.1s
LSTM 256 19.3s
LSTM 512 20.6s
LSTM 777 25.3s
LSTM 2024 27.1s
LSTM-TPE 42 11.0s
LSTM-TPE 7 10.1s
LSTM-TPE 13 8.6s
LSTM-TPE 21 8.1s
LSTM-TPE 99 9.3s
LSTM-TPE 123 14.8s
LSTM-TPE 256 16.8s
LSTM-TPE 512 14.9s
LSTM-TPE 777 14.5s
LSTM-TPE 2024 12.4s
GRU 42 9.4s
GRU 7 7.4s
GRU 13 21.4s
GRU 21 17.9s
GRU 99 18.7s
GRU 123 14.6s
GRU 256 14.6s
GRU 512 11.7s
GRU 777 13.4s
GRU 2024 27.6s
GRU-TPE 42 22.3s
GRU-TPE 7 15.8s
GRU-TPE 13 11.7s
GRU-TPE 21 16.1s
GRU-TPE 99 23.7s
GRU-TPE 123 24.0s
GRU-TPE 256 11.7s
GRU-TPE 512 8.9s
GRU-TPE 777 23.2s
GRU-TPE 2024 15.2s
GRU-TPE-noDJIA 42 18.5s
GRU-TPE-noDJIA 7 14.0s
GRU-TPE-noDJIA 13 11.7s
GRU-TPE-noDJIA 21 31.9s
GRU-TPE-noDJIA 99 11.9s
GRU-TPE-noDJIA 123 21.3s
GRU-TPE-noDJIA 256 13.5s
GRU-TPE-noDJIA 512 15.2s
GRU-TPE-noDJIA 777 25.5s
GRU-TPE-noDJIA 2024 15.3s
GRU-TPE-return 42 6.3s
GRU-TPE-return 7 6.2s
GRU-TPE-return 13 5.9s
GRU-TPE-return 21 11.2s
GRU-TPE-return 99 6.7s
GRU-TPE-

## 4. Pembanding ARIMA (script 23)
Orde dipilih dengan AIC pada data latih saja, lalu parameternya dibekukan untuk peramalan satu langkah di data uji. Hasil: `arima_predictions.csv` dan `arima_orders.csv`.

In [7]:
import sys
!{sys.executable} -m pip install statsmodels



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import itertools
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")
df = load_series()
y = df["IHSG_Close"].values
SPLITS = {"main": (2069, 2587), "wf1": (2069, 2242), "wf2": (2242, 2415), "wf3": (2415, 2587)}

rows, orders = [], []
for name, (train_end, test_end) in SPLITS.items():
    best = None
    for p, q, trend in itertools.product(range(4), range(4), ["n", "t"]):
        try:
            res = ARIMA(y[:train_end], order=(p, 1, q), trend=trend).fit()
        except Exception:
            continue
        if best is None or res.aic < best[0]:
            best = (res.aic, (p, 1, q), trend, res)
    aic, order, trend, res = best
    full = res.apply(y[:test_end])                      # same parameters, filtered over the full series
    pred = full.get_prediction(start=train_end, end=test_end - 1).predicted_mean
    orders.append({"split": name, "order": str(order), "trend": trend, "aic": aic})
    rows.append(pd.DataFrame({"split": name, "Date": df["Date"].iloc[train_end:test_end].dt.strftime("%Y-%m-%d"),
                              "actual": y[train_end:test_end], "pred": np.asarray(pred)}))
    print(name, order, trend, "AIC %.1f" % aic, flush=True)

pd.concat(rows).to_csv("arima_predictions.csv", index=False)
pd.DataFrame(orders).to_csv("arima_orders.csv", index=False)

main (3, 1, 3) n AIC 22389.0
wf1 (3, 1, 3) n AIC 22389.0
wf2 (3, 1, 3) n AIC 24262.1
wf3 (0, 1, 3) n AIC 26406.7


## 5. Evaluasi walk-forward (script 24)
Periode uji dibagi tiga blok, dan setiap jaringan dilatih ulang sebelum tiap blok dengan tiga seed. Hasil ditambahkan ke `walkforward_predictions.csv`, jadi hapus file itu dulu jika ingin mengulang dari awal.

In [9]:
import os
import sys
import time
import pandas as pd

ORIGINS = [(2069, 2242), (2242, 2415), (2415, 2587)]
df = load_series()
models = ["LSTM", "LSTM-TPE", "GRU", "GRU-TPE"]

def config_for(name):
    if name == "LSTM-TPE":
        res = pd.read_csv("optuna_lstm_results.csv")
        b = res.loc[res["value"].idxmin()]
        return dict(cell="LSTM", units=int(b["params_units"]), dropout=float(b["params_dropout_rate"]),
                    lr=float(b["params_learning_rate"]), batch=int(b["params_batch_size"]))
    return CONFIGS[name]

out_file = "walkforward_predictions.csv"
for name in models:
    cfg = config_for(name)
    for k, (start, end) in enumerate(ORIGINS, 1):
        X_tr, y_tr, X_te, inverse = make_level_windows(df, start, start, end)
        for seed in SEEDS[:3]:
            t0 = time.time()
            pred, _ = train_predict(cfg, X_tr, y_tr, X_te, seed)
            pd.DataFrame({"model": name, "block": k, "seed": seed,
                          "Date": df["Date"].iloc[start:end].dt.strftime("%Y-%m-%d"),
                          "actual": df["IHSG_Close"].iloc[start:end].values, "pred": inverse(pred)}
                         ).to_csv(out_file, mode="a", header=not os.path.exists(out_file), index=False)
            print(name, k, seed, "%.1fs" % (time.time() - t0), flush=True)

LSTM 1 42 26.2s
LSTM 1 7 8.2s
LSTM 1 13 26.8s
LSTM 2 42 28.5s
LSTM 2 7 27.6s
LSTM 2 13 17.8s
LSTM 3 42 29.4s
LSTM 3 7 29.5s
LSTM 3 13 31.1s
LSTM-TPE 1 42 11.4s
LSTM-TPE 1 7 10.8s
LSTM-TPE 1 13 9.1s
LSTM-TPE 2 42 8.5s
LSTM-TPE 2 7 16.6s
LSTM-TPE 2 13 18.0s
LSTM-TPE 3 42 14.0s
LSTM-TPE 3 7 22.8s
LSTM-TPE 3 13 22.1s
GRU 1 42 10.0s
GRU 1 7 7.6s
GRU 1 13 22.1s
GRU 2 42 14.4s
GRU 2 7 12.0s
GRU 2 13 13.4s
GRU 3 42 26.1s
GRU 3 7 19.7s
GRU 3 13 22.7s
GRU-TPE 1 42 22.2s
GRU-TPE 1 7 15.7s
GRU-TPE 1 13 11.6s
GRU-TPE 2 42 10.7s
GRU-TPE 2 7 12.5s
GRU-TPE 2 13 11.6s
GRU-TPE 3 42 16.1s
GRU-TPE 3 7 18.9s
GRU-TPE 3 13 18.7s


## 6. Rekap ketahanan hasil (script 25)
Tabel 4, 5, 9, dan 10 di artikel berasal dari sel ini.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))


def dm_test(actual, pred_a, pred_b):
    """Positive statistic: model B is more accurate than model A (squared-error loss, h = 1)."""
    d = (actual - pred_a) ** 2 - (actual - pred_b) ** 2
    s = d.mean() / np.sqrt(np.var(d) / len(d))
    return s, 2 * (1 - stats.t.cdf(abs(s), df=len(d) - 1))


def holm(pvalues):
    p = np.asarray(pvalues, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (len(p) - rank) * p[idx])
        adj[idx] = min(1.0, running)
    return adj


# ------------------------------------------------------------------ 1. original run
base = pd.read_csv("baseline_predictions.csv")
opt = pd.read_csv("optimized_gru_predictions.csv")
train = pd.read_csv("train_data.csv")
arima = pd.read_csv("arima_predictions.csv")
y = base["Actual_IHSG"].values
prev = np.r_[train["IHSG_Close"].values[-1], y[:-1]]
orig = {"Random walk": prev,
        "ARIMA": arima.loc[arima["split"] == "main", "pred"].values,
        "LSTM": base["LSTM_Prediction"].values,
        "GRU": base["GRU_Prediction"].values,
        "GRU-TPE": opt["Predicted_IHSG"].values}
_ms = pd.read_csv("multiseed_predictions.csv")
orig["LSTM-TPE"] = _ms[(_ms["model"] == "LSTM-TPE") & (_ms["seed"] == 42)]["pred"].values[:518]
rw = rmse(y - prev)
rows = []
for k, p in orig.items():
    e = y - p
    rows.append({"Method": k, "RMSE": rmse(e), "MAE": np.mean(np.abs(e)), "MAPE (%)": 100 * np.mean(np.abs(e / y)),
                 "R2": 1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2), "U": rmse(e) / rw})
print("ORIGINAL RUN (seed 42) WITH BENCHMARKS")
print(pd.DataFrame(rows).round(4).to_string(index=False))

pairs = [("LSTM", "LSTM-TPE"), ("GRU", "GRU-TPE"), ("LSTM-TPE", "GRU-TPE"),
         ("Random walk", "ARIMA"), ("Random walk", "LSTM"), ("Random walk", "LSTM-TPE"),
         ("Random walk", "GRU"), ("Random walk", "GRU-TPE")]
dm = [(a, b) + dm_test(y, orig[a], orig[b]) for a, b in pairs]
dm = pd.DataFrame(dm, columns=["A", "B", "DM", "p"])
dm["p_Holm"] = holm(dm["p"])
print("\nDIEBOLD-MARIANO TESTS, ORIGINAL RUN (Holm-adjusted over all pairs)")
print(dm.round(4).to_string(index=False))
dm.to_csv("article_dm_holm.csv", index=False)

# ------------------------------------------------------------------ 2. ten seeds
ms = pd.read_csv("multiseed_predictions.csv")
ms_rows = []
per_seed = []
for (model, seed), g in ms.groupby(["model", "seed"], sort=False):
    a = g["actual"].values[:518]
    p = g["pred"].values[:518]
    assert np.allclose(a, y)
    s, pv = dm_test(a, prev, p)
    corr_dir = (np.sign((p - prev)[1:]) == np.sign((a - prev)[1:])).mean() * 100
    per_seed.append({"model": model, "seed": seed, "RMSE": rmse(a - p), "U": rmse(a - p) / rw,
                     "DM_vs_RW": s, "p_vs_RW": pv, "mean_abs_change": np.mean(np.abs(p - prev)), "DA": corr_dir})
per_seed = pd.DataFrame(per_seed)
per_seed.to_csv("article_multiseed_per_seed.csv", index=False)
order = ["LSTM", "LSTM-TPE", "GRU", "GRU-TPE", "GRU-TPE-noDJIA", "GRU-TPE-return"]
summary = (per_seed.groupby("model")
           .agg(RMSE_mean=("RMSE", "mean"), RMSE_sd=("RMSE", "std"), U_mean=("U", "mean"),
                U_min=("U", "min"), U_max=("U", "max"),
                seeds_beating_RW=("U", lambda u: int((u < 1).sum())),
                seeds_sig_worse=("p_vs_RW", lambda p: None),
                change_mean=("mean_abs_change", "mean"), DA_mean=("DA", "mean"))
           .reindex([m for m in order if m in set(per_seed["model"])]))
summary["seeds_sig_worse"] = [int(((per_seed.model == m) & (per_seed.DM_vs_RW < 0) & (per_seed.p_vs_RW < 0.05)).sum())
                              for m in summary.index]
summary["seeds_sig_better"] = [int(((per_seed.model == m) & (per_seed.DM_vs_RW > 0) & (per_seed.p_vs_RW < 0.05)).sum())
                               for m in summary.index]
print("\nTEN-SEED SUMMARY (main split; random walk RMSE = %.3f)" % rw)
print(summary.round(3).to_string())
summary.to_csv("article_multiseed_summary.csv")

print("\nPAIRED COMPARISONS ACROSS SEEDS (RMSE of A minus RMSE of B)")
piv = per_seed.pivot(index="seed", columns="model", values="RMSE")
for a, b in [("GRU", "GRU-TPE"), ("LSTM", "LSTM-TPE"), ("LSTM-TPE", "GRU-TPE"),
             ("GRU-TPE", "GRU-TPE-noDJIA"), ("GRU-TPE", "GRU-TPE-return")]:
    if a in piv and b in piv:
        diff = piv[a] - piv[b]
        w = stats.wilcoxon(piv[a], piv[b])
        print("%-10s vs %-15s mean diff %7.3f  sd %6.3f  wins %d/10  Wilcoxon p = %.4f"
              % (a, b, diff.mean(), diff.std(), int((diff > 0).sum()), w.pvalue))

# ------------------------------------------------------------------ 3. walk-forward
wf = pd.read_csv("walkforward_predictions.csv")
wf_arima = arima[arima["split"].isin(["wf1", "wf2", "wf3"])]
assert np.allclose(wf_arima["actual"].values, y)
wf_rows = [{"Method": "Random walk", "RMSE_mean": rw, "RMSE_sd": 0.0, "U_mean": 1.0},
           {"Method": "ARIMA (re-estimated per block)", "RMSE_mean": rmse(y - wf_arima["pred"].values),
            "RMSE_sd": 0.0, "U_mean": rmse(y - wf_arima["pred"].values) / rw}]
for model, g in wf.groupby("model", sort=False):
    r = []
    for seed, gs in g.groupby("seed"):
        gs = gs.sort_values("Date")
        actual_wf = gs["actual"].values[:518]
        pred_wf = gs["pred"].values[:518]
        
        assert np.allclose(actual_wf, y)
        r.append(rmse(y - pred_wf))
    wf_rows.append({"Method": model, "RMSE_mean": np.mean(r), "RMSE_sd": np.std(r, ddof=1), "U_mean": np.mean(r) / rw})
wf_table = pd.DataFrame(wf_rows)
s_ar, p_ar = dm_test(y, prev, wf_arima["pred"].values)
print("DM random walk vs walk-forward ARIMA: %.3f (p = %.4f)" % (s_ar, p_ar))
print("\nWALK-FORWARD (three expanding-window blocks, three seeds per network)")
print(wf_table.round(3).to_string(index=False))
wf_table.to_csv("article_walkforward.csv", index=False)

ORIGINAL RUN (seed 42) WITH BENCHMARKS
     Method    RMSE     MAE  MAPE (%)     R2      U
Random walk 72.7421 53.5723    0.7438 0.9814 1.0000
      ARIMA 73.2149 53.7362    0.7463 0.9812 1.0065
       LSTM 84.7261 63.7552    0.8858 0.9748 1.1647
        GRU 81.4721 60.9238    0.8468 0.9767 1.1200
    GRU-TPE 73.7028 54.4745    0.7555 0.9809 1.0132
   LSTM-TPE 82.6852 60.0723    0.8389 0.9760 1.1367

DIEBOLD-MARIANO TESTS, ORIGINAL RUN (Holm-adjusted over all pairs)
          A        B      DM      p  p_Holm
       LSTM LSTM-TPE  1.7801 0.0757  0.2270
        GRU  GRU-TPE  3.5214 0.0005  0.0028
   LSTM-TPE  GRU-TPE  3.2440 0.0013  0.0050
Random walk    ARIMA -1.1025 0.2707  0.2707
Random walk     LSTM -4.1353 0.0000  0.0003
Random walk LSTM-TPE -3.5145 0.0005  0.0028
Random walk      GRU -3.7006 0.0002  0.0017
Random walk  GRU-TPE -1.7706 0.0772  0.2270

TEN-SEED SUMMARY (main split; random walk RMSE = 72.742)
                RMSE_mean  RMSE_sd  U_mean  U_min  U_max  seeds_beating_RW 